<a href="https://colab.research.google.com/github/FridaOyucho/HTS-Model-/blob/main/HTSModelDataPrep_functions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HTS Model Data Prep (function-based)

Every section of the original `HTSModelDataPrep26082026` script is now a function.

**Stages**
1. Combine and filter original tables → `run_stage1_combine_and_filter()`
2. Data cleaning and feature generation → `run_stage2_clean_and_engineer()`
3. GIS join, train/val/test split, imputation → `run_stage3_impute_and_split()`
4. Positivity over time → `run_stage4_positivity_over_time()`

Run the cells top to bottom to define everything, then run the **Run the pipeline** cells at the end (each stage can be run on its own).

In [ ]:
# Run once in Colab (needed for Stage 3 GIS join)
!pip install pyreadr

## Setup: imports

In [ ]:
import random
import pickle
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:                                   # plain Python fallback
    def display(obj):
        print(obj)

## Configuration

File paths, column lists and settings. Edit `PATHS` if your file names differ (e.g. Colab-renamed `hts (4).csv`).

In [ ]:
VERBOSE = True

SEED = 2231

KEYS = ["SiteCode", "PatientPk"]

TARGET = "FinalTestResult"

ELIGIBILITY_START = "2025-04-01"

ELIGIBILITY_END = "2026-06-30"

PATHS = {
    "eligibility": "eligibility.csv",
    "tests": "tests.csv",
    "xwalk": "ActiveEMRSites_07152026.csv",
    "clients": "clients.csv",
    "merged": "merged.csv",
    "hts_raw": "hts.csv",                 # output of Stage 1 / input of Stage 2
    "hts_cleaned": "hts_cleaned.csv",     # output of Stage 2 / input of Stage 3
    "gis": "gis_features_iit.rds",
}

TESTS_COLS = ["SiteCode", "PatientPk", "EverTestedForHiv", "MonthsSinceLastTest",
              "FinalTestResult", "ClientTestedAs", "CoupleDiscordant",
              "PriorityPopulationType"]

ELIGIBILITY_COLS = [
    "SiteCode", "PatientPk", "VisitDate", "PopulationType", "KeyPopulation",
    "PriorityPopulation", "IsHealthWorker", "RelationshipWithContact",
    "TestedHIVBefore", "ResultOfHIV", "EverHadSex", "SexuallyActive",
    "NewPartner", "PartnerHIVStatus", "MultiplePartners", "NumberOfPartners",
    "AlcoholSex", "MoneySex", "CondomBurst", "UnknownStatusPartner",
    "KnownStatusPartner", "Pregnant", "BreastfeedingMother",
    "ExperiencedViolenceScreening", "CurrentlyOnPrep", "TraditionalProcedures",
    "MothersStatus", "ResultOfHIVSelf", "ScreenedTB", "TBStatus",
    "ReceivedServices", "ViolenceScreeningType", "Disability", "DisabilityType",
]

XWALK_COLS = ["MFL_Code", "Facility_Name", "County", "SubCounty", "Owner",
              "Latitude", "Longitude", "SDP", "SDP_Agency", "EMR", "Project",
              "KEPH_Level", "Ward"]

HIGH_MISSING_DROP = ["Project", "SDP_Agency", "Owner", "Disability",
                     "DisabilityType", "RelationshipWithContact"]

FINAL_DROP_COLS = [
    "Dob", "DOB", "present", "cohort", "VDate", "CurrentlyOnPep",
    "DateTestedProvider", "Disability", "DisabilityType", "LoadDate_x",
    "KMPDC_reg_no", "ReceivedPEP", "EMR_Status", "HTS_Use", "HTS_Deployment",
    "Project", "LoadDate_y", "InfrastructureType", "PriorityPopulation-grouped",
    "PriorityPopulation_grouped", "PriorityPopulation-group",
    "PriorityPopulation_group", "TBStatus-grouped", "ScreenedTB..",
    "KEPH_Level", "Ward",
]

VARIANCE_CHECK_COLS = [
    "PopulationType", "KeyPopulation", "PriorityPopulation", "IsHealthWorker",
    "RelationshipWithContact", "TestedHIVBefore", "ResultOfHIV", "EverHadSex",
    "SexuallyActive", "NewPartner", "PartnerHIVStatus", "MultiplePartners",
    "NumberOfPartners", "AlcoholSex", "MoneySex", "CondomBurst",
    "UnknownStatusPartner", "KnownStatusPartner", "Pregnant",
    "BreastfeedingMother", "ExperiencedViolenceScreening", "CurrentlyOnPrep",
    "TraditionalProcedures", "MothersStatus", "ResultOfHIVSelf", "ScreenedTB",
    "TBStatus", "ReceivedServices", "ViolenceScreeningType", "Sex",
    "MaritalStatus", "PatientDisabled", "EverTestedForHiv",
    "MonthsSinceLastTest", "FinalTestResult", "PriorityPopulationType",
    "CoupleDiscordant", "Age", "ReceivedPREP", "ReceivedTB", "ReceivedSTI",
    "GBVSexual", "GBVPhysical", "GBVEmotional", "dayofweek",
]

## GENERIC HELPERS

In [ ]:
def print_shape(df, name):
    """Print the row/column count of a table."""
    rows, cols = df.shape
    print(f"There are {rows} rows and {cols} columns in the {name} table")

In [ ]:
def show_counts(df, col):
    """Display value counts (incl. NaN) for a column when VERBOSE."""
    if VERBOSE:
        display(df[col].value_counts(dropna=False))

In [ ]:
def positivity_table(df, by, show=True):
    """Crosstab of `by` (a column or list of columns) against the test result,
    with a PositivityRate (%) column."""
    index = df[by] if isinstance(by, str) else [df[c] for c in by]
    tab = pd.crosstab(index, df[TARGET], dropna=False)
    tab["PositivityRate"] = tab["POSITIVE"] / (tab["POSITIVE"] + tab["NEGATIVE"]) * 100
    if show and VERBOSE:
        display(tab)
    return tab

In [ ]:
def crosstab_check(df, row_col, col_col, show=True, **kwargs):
    """Simple crosstab between two columns (used for quick consistency checks)."""
    tab = pd.crosstab(df[row_col], df[col_col], dropna=False, **kwargs)
    if show and VERBOSE:
        display(tab)
    return tab

In [ ]:
def select_labels(df, conditions, choices):
    """np.select wrapper: unmatched rows become NaN (the original 'None' -> NaN pattern)."""
    out = np.select(conditions, choices, default="None")
    return pd.Series(out, index=df.index).replace({"None": np.nan})

In [ ]:
def recode_yes_no(df, col, nr_rules=(), allow_declined=True,
                  yes_label="YES", no_label="NO", as_str=False):
    """
    Shared recoding used by most YES/NO screening questions.

    Order of rules (first match wins, same as the original np.select blocks):
        DECLINED TO ANSWER -> DECLINED
        each rule in nr_rules (callables taking df) -> NR
        YES -> YES
        NO  -> NO
        anything else -> NaN
    """
    df[col] = df[col].astype(str).str.upper() if as_str else df[col].str.upper()
    conditions, choices = [], []
    if allow_declined:
        conditions.append(df[col].eq("DECLINED TO ANSWER"))
        choices.append("DECLINED")
    for rule in nr_rules:
        conditions.append(rule(df))
        choices.append("NR")
    conditions += [df[col].eq("YES"), df[col].eq("NO")]
    choices += [yes_label, no_label]
    df[col] = select_labels(df, conditions, choices)
    return df

In [ ]:
def count_yes_without_sex(df, col):
    """Count rows where `col` is YES but EverHadSex is NO (consistency check)."""
    n = df[(df[col] == "YES") & (df["EverHadSex"] == "NO")].shape[0]
    print(f"Number of rows where {col} is 'YES' and EverHadSex is 'NO': {n}")
    return n

In [ ]:
def save_csv(df, path, download=False):
    """Save a DataFrame to CSV; optionally trigger a Colab download."""
    df.to_csv(path, index=False)
    if download:
        download_file(path)

In [ ]:
def download_file(path):
    """Download a file in Google Colab (no-op elsewhere)."""
    try:
        from google.colab import files
        files.download(path)
    except ImportError:
        print(f"(Not running in Colab - file saved locally as {path})")

## STAGE 1 - COMBINE AND FILTER ORIGINAL TABLES

In [ ]:
def load_tables(paths=PATHS):
    """Load eligibility, tests, xwalk and clients tables and print their shapes."""
    eligibility = pd.read_csv(paths["eligibility"], low_memory=False)
    tests = pd.read_csv(paths["tests"], low_memory=False)
    xwalk = pd.read_csv(paths["xwalk"], low_memory=False)
    clients = pd.read_csv(paths["clients"], low_memory=False)
    for name, df in [("eligibility", eligibility), ("tests", tests),
                     ("xwalk", xwalk), ("clients", clients)]:
        print_shape(df, name)
    return eligibility, tests, xwalk, clients

In [ ]:
def prepare_tests(tests):
    """Keep POSITIVE/NEGATIVE results only, select columns, drop duplicates."""
    tests = tests.copy()
    if VERBOSE:
        display(tests.columns)
    tests[TARGET] = tests[TARGET].str.upper()
    tests = tests[tests[TARGET].isin(["POSITIVE", "NEGATIVE"])]
    tests = tests[TESTS_COLS].drop_duplicates(subset=KEYS)
    print_shape(tests, "tests")
    show_counts(tests, TARGET)
    return tests

In [ ]:
def prepare_clients(clients):
    """Keep all client columns, remove duplicates on SiteCode + PatientPk."""
    if VERBOSE:
        display(clients.columns)
    clients = clients.drop_duplicates(subset=KEYS)
    print_shape(clients, "clients")
    return clients

In [ ]:
def prepare_eligibility(eligibility, start=ELIGIBILITY_START, end=ELIGIBILITY_END):
    """Convert VisitDate, filter to the study window, select columns, drop duplicates."""
    eligibility = eligibility.copy()
    print(eligibility["VisitDate"].dtype)
    eligibility["VisitDate"] = pd.to_datetime(eligibility["VisitDate"].astype(str).str[:10])
    eligibility = eligibility[(eligibility["VisitDate"] >= start) &
                              (eligibility["VisitDate"] <= end)]
    print_shape(eligibility, "eligibility")
    eligibility = eligibility[ELIGIBILITY_COLS].drop_duplicates(subset=KEYS)
    print_shape(eligibility, "eligibility")
    return eligibility

In [ ]:
def prepare_xwalk(xwalk):
    """Select facility crosswalk columns and drop duplicate MFL codes."""
    xwalk = xwalk[XWALK_COLS].drop_duplicates(subset=["MFL_Code"])
    print_shape(xwalk, "xwalk")
    return xwalk

In [ ]:
def merge_tables(eligibility, clients, tests, xwalk):
    """Join eligibility + clients + tests on SiteCode/PatientPk, then add facility info."""
    merged = pd.merge(eligibility, clients, on=KEYS, how="left")
    merged = pd.merge(merged, tests, on=KEYS, how="left")
    merged = pd.merge(merged, xwalk, left_on=["SiteCode"], right_on=["MFL_Code"], how="left")
    print_shape(merged, "merged")
    return merged

In [ ]:
def exclude_known_positives(hts):
    """Exclude records whose ResultOfHIV is already POSITIVE."""
    hts = hts.copy()
    hts["ResultOfHIV"] = hts["ResultOfHIV"].str.upper()
    hts = hts[hts["ResultOfHIV"] != "POSITIVE"]
    print_shape(hts, "HTS")
    return hts

In [ ]:
def check_patient_disabled_vs_result(clients, tests):
    """Quick check kept from the original (index-aligned crosstab of clients vs tests).
    NOTE: this aligns on row index, not on SiteCode/PatientPk, so treat it as indicative
    only - the PatientDisabled positivity table in Stage 2 is the reliable one."""
    return pd.crosstab(clients["PatientDisabled"], tests[TARGET], dropna=False)

In [ ]:
def report_high_missingness(hts, threshold=50):
    """Print variables with more than `threshold`% missing values."""
    missing_percent = hts.isnull().mean() * 100
    high_missing = missing_percent[missing_percent > threshold].sort_values(ascending=False)
    print(high_missing)
    return high_missing

In [ ]:
def drop_high_missing_variables(hts, variables_to_drop=HIGH_MISSING_DROP):
    """Drop variables with very high missingness / no analytical use."""
    return hts.drop(columns=variables_to_drop, errors="ignore")

In [ ]:
def run_stage1_combine_and_filter(paths=PATHS, download=False):
    """Run all of Stage 1 and save merged.csv and hts.csv."""
    eligibility, tests, xwalk, clients = load_tables(paths)
    tests = prepare_tests(tests)
    clients = prepare_clients(clients)
    eligibility = prepare_eligibility(eligibility)
    xwalk = prepare_xwalk(xwalk)

    merged = merge_tables(eligibility, clients, tests, xwalk)
    save_csv(merged, paths["merged"])

    hts = merged.copy()
    print_shape(hts, "hts")
    hts = exclude_known_positives(hts)
    report_high_missingness(hts)
    hts = drop_high_missing_variables(hts)
    save_csv(hts, paths["hts_raw"], download=download)
    return hts

## STAGE 2 - DATA CLEANING AND FEATURE GENERATION

In [ ]:
def load_hts_raw(path=PATHS["hts_raw"]):
    """Load the Stage 1 output (rename to match if Colab added a suffix, e.g. 'hts (4).csv')."""
    hts = pd.read_csv(path, low_memory=False)
    print_shape(hts, "hts")
    return hts

### Demographic features

In [ ]:
def add_age_features(hts):
    """Create VDate, DOB, Age (years); keep 0 < Age < 100; add Adult/Child cohort."""
    hts = hts.copy()
    hts["VDate"] = pd.to_datetime(hts["VisitDate"].astype(str).str[:10])
    hts["DOB"] = pd.to_datetime(hts["Dob"].astype(str).str[:10], errors="coerce")
    hts["Age"] = (hts["VDate"] - hts["DOB"]).dt.days // 365
    hts = hts[(hts["Age"] > 0) & (hts["Age"] < 100)].copy()
    hts["cohort"] = np.where(hts["Age"] >= 15, "Adult", "Child")
    print_shape(hts, "hts (after age filtering)")
    return hts

In [ ]:
def clean_sex(hts):
    """Standardise Sex to M / F and show positivity by sex."""
    hts["Sex"] = (hts["Sex"].astype(str).str.upper()
                  .str.replace("FEMALE", "F").str.replace("MALE", "M"))
    positivity_table(hts, "Sex")           # positivity is higher in males
    return hts

In [ ]:
def drop_missing_test_result(hts):
    """Keep only rows that have a test result (the model target)."""
    show_counts(hts, TARGET)
    hts = hts.dropna(subset=[TARGET])
    show_counts(hts, TARGET)
    return hts

In [ ]:
def clean_marital_status(hts):
    """Group MaritalStatus into MINOR / UNKNOWN / DIVORCED / POLYGAMOUS / SINGLE / MARRIED / WIDOWED."""
    hts["MaritalStatus"] = hts["MaritalStatus"].str.upper().replace("OTHER", "UNKNOWN")
    ms = hts["MaritalStatus"]
    conditions = [
        (hts["Age"] < 15) & ms.notna(),
        ms.eq("UNKNOWN"),
        ms.isin(["DIVORCED", "SEPARATED"]),
        ms.eq("MARRIED POLYGAMOUS"),
        ms.isin(["SINGLE", "NO"]),
        ms.isin(["MARRIED MONOGAMOUS", "COHABITING", "YES"]),
        ms.eq("WIDOWED"),
    ]
    choices = ["MINOR", "UNKNOWN", "DIVORCED", "POLYGAMOUS", "SINGLE", "MARRIED", "WIDOWED"]
    hts["MaritalStatus"] = select_labels(hts, conditions, choices)
    show_counts(hts, "MaritalStatus")
    positivity_table(hts, "MaritalStatus")             # higher in divorced/widowed
    positivity_table(hts, ["EMR", "MaritalStatus"])    # distribution of missing values by EMR
    crosstab_check(hts, "MaritalStatus", "EverHadSex")
    return hts

### Clinical risk features

In [ ]:
def clean_pregnant(hts):
    """Pregnant: NR for <10 yrs, males, >50 yrs."""
    recode_yes_no(hts, "Pregnant", nr_rules=[
        lambda d: (d["Age"] < 10) & d["Pregnant"].notna(),
        lambda d: (d["Sex"].str.upper() == "M") & d["Pregnant"].notna(),
        lambda d: (d["Age"] > 50) & d["Pregnant"].notna(),
    ])
    show_counts(hts, "Pregnant")
    crosstab_check(hts, "Pregnant", "EMR")
    positivity_table(hts, "Pregnant")                  # low among pregnant women
    return hts

In [ ]:
def clean_breastfeeding_mother(hts):
    """BreastfeedingMother: NR for <10 yrs, males, >50 yrs."""
    recode_yes_no(hts, "BreastfeedingMother", nr_rules=[
        lambda d: (d["Age"] < 10) & d["BreastfeedingMother"].notna(),
        lambda d: (d["Sex"].str.upper() == "M") & d["BreastfeedingMother"].notna(),
        lambda d: (d["Age"] > 50) & d["BreastfeedingMother"].notna(),
    ])
    show_counts(hts, "BreastfeedingMother")
    crosstab_check(hts, "BreastfeedingMother", "EMR")
    positivity_table(hts, "BreastfeedingMother")       # low among breastfeeding women
    return hts

### HIV testing history

In [ ]:
def clean_tested_hiv_before(hts):
    """TestedHIVBefore: standardise to upper case."""
    show_counts(hts, "TestedHIVBefore")
    hts["TestedHIVBefore"] = hts["TestedHIVBefore"].str.upper()
    positivity_table(hts, "TestedHIVBefore")           # higher among ever-tested
    return hts

In [ ]:
def clean_months_since_last_test(hts):
    """Bin MonthsSinceLastTest into NR / LASTSIXMONTHS / SEVENTOTWELVE / ONETOTWOYEARS / MORETHANTWOYEARS."""
    hts["EverTestedForHiv"] = hts["EverTestedForHiv"].str.upper()
    hts["MonthsSinceLastTest"] = pd.to_numeric(hts["MonthsSinceLastTest"], errors="coerce")
    m = hts["MonthsSinceLastTest"]
    conditions = [
        hts["EverTestedForHiv"] != "YES",
        (m >= 0) & (m <= 6),
        (m >= 7) & (m <= 12),
        (m >= 13) & (m <= 24),
        m > 24,
    ]
    choices = ["NR", "LASTSIXMONTHS", "SEVENTOTWELVE", "ONETOTWOYEARS", "MORETHANTWOYEARS"]
    hts["MonthsSinceLastTest"] = select_labels(hts, conditions, choices)
    positivity_table(hts, "MonthsSinceLastTest")       # higher for >2 years ago
    return hts

### Behaviour risk features

In [ ]:
def clean_new_partner(hts):
    """NewPartner: NR if never had sex or <15 yrs. Also standardises EverHadSex to upper case
    (needed by this and the following sexual-behaviour variables)."""
    hts["EverHadSex"] = hts["EverHadSex"].str.upper()
    recode_yes_no(hts, "NewPartner", nr_rules=[
        lambda d: d["EverHadSex"].eq("NO"),
        lambda d: (d["Age"] < 15) & d["NewPartner"].notna(),
    ])
    show_counts(hts, "NewPartner")
    positivity_table(hts, "NewPartner")
    return hts

In [ ]:
def check_sexually_active_vs_new_partner(hts):
    """Review check: sexually-inactive clients who report a new partner."""
    sa = hts["SexuallyActive"].astype(str).str.upper()
    tab = pd.crosstab(sa, hts["NewPartner"])
    if VERBOSE:
        display(tab)
    return tab

In [ ]:
def clean_multiple_partners(hts):
    """MultiplePartners: NR if never had sex or <15 yrs."""
    recode_yes_no(hts, "MultiplePartners", allow_declined=False, nr_rules=[
        lambda d: d["EverHadSex"].eq("NO"),
        lambda d: (d["Age"] < 15) & d["MultiplePartners"].notna(),
    ])
    show_counts(hts, "MultiplePartners")
    positivity_table(hts, "MultiplePartners")
    return hts

In [ ]:
def check_multiple_partners_vs_sexually_active(hts):
    """Review check: MultiplePartners = YES but SexuallyActive = NO, plus cohort breakdown."""
    sa = hts["SexuallyActive"].astype(str).str.upper()
    n = ((hts["MultiplePartners"] == "YES") & (sa == "NO")).sum()
    print(f"Number of rows where MultiplePartners is 'YES' and SexuallyActive is 'NO': {n}")
    tab = pd.crosstab(index=[hts["cohort"], hts["MultiplePartners"]],
                      columns=sa, margins=True, dropna=False)
    if VERBOSE:
        display(tab)
    return n, tab

In [ ]:
def clean_money_sex(hts):
    """MoneySex: NR if <15 yrs or never had sex."""
    recode_yes_no(hts, "MoneySex", nr_rules=[
        lambda d: (d["Age"] < 15) & d["MoneySex"].notna(),
        lambda d: d["EverHadSex"].eq("NO"),
    ])
    show_counts(hts, "MoneySex")
    positivity_table(hts, "MoneySex")
    count_yes_without_sex(hts, "MoneySex")
    return hts

In [ ]:
def clean_alcohol_sex(hts):
    """AlcoholSex: ALWAYS / SOMETIMES / NEVER, NR if <15 yrs or never had sex."""
    show_counts(hts, "AlcoholSex")
    hts["AlcoholSex"] = hts["AlcoholSex"].str.upper()
    conditions = [
        hts["AlcoholSex"].eq("DECLINED TO ANSWER"),
        (hts["Age"] < 15) & hts["AlcoholSex"].notna(),
        hts["EverHadSex"].eq("NO"),
        hts["AlcoholSex"].eq("ALWAYS"),
        hts["AlcoholSex"].eq("SOMETIMES"),
        hts["AlcoholSex"].eq("NOT AT ALL"),
    ]
    choices = ["DECLINED", "NR", "NR", "ALWAYS", "SOMETIMES", "NEVER"]
    hts["AlcoholSex"] = select_labels(hts, conditions, choices)
    show_counts(hts, "AlcoholSex")
    positivity_table(hts, "AlcoholSex")
    return hts

In [ ]:
def clean_condom_burst(hts):
    """CondomBurst: NR if <=9 yrs or never had sex."""
    show_counts(hts, "CondomBurst")
    recode_yes_no(hts, "CondomBurst", nr_rules=[
        lambda d: (d["Age"] <= 9) & d["CondomBurst"].notna(),
        lambda d: d["EverHadSex"].eq("NO"),
    ])
    show_counts(hts, "CondomBurst")
    positivity_table(hts, "CondomBurst")
    count_yes_without_sex(hts, "CondomBurst")
    return hts

### Partner risk features

In [ ]:
def clean_partner_hiv_status(hts):
    """PartnerHIVStatus: POSITIVE / NEGATIVE / UNKNOWN, NR if <15 yrs or never had sex."""
    show_counts(hts, "PartnerHIVStatus")
    hts["PartnerHIVStatus"] = (hts["PartnerHIVStatus"].str.upper()
                               .replace("NEGATIVE", "HIV NEGATIVE")
                               .replace("POSITIVE", "HIV POSITIVE"))
    conditions = [
        hts["PartnerHIVStatus"].eq("DECLINED TO ANSWER"),
        (hts["Age"] < 15) & hts["PartnerHIVStatus"].notna(),
        hts["EverHadSex"].eq("NO"),
        hts["PartnerHIVStatus"].eq("HIV POSITIVE"),
        hts["PartnerHIVStatus"].eq("HIV NEGATIVE"),
        hts["PartnerHIVStatus"].eq("UNKNOWN"),
    ]
    choices = ["DECLINED", "NR", "NR", "POSITIVE", "NEGATIVE", "UNKNOWN"]
    hts["PartnerHIVStatus"] = select_labels(hts, conditions, choices)
    show_counts(hts, "PartnerHIVStatus")
    positivity_table(hts, "PartnerHIVStatus")
    print_shape(hts, "hts")
    return hts

In [ ]:
def clean_unknown_status_partner(hts):
    """UnknownStatusPartner: unprotected sex with a partner of unknown status; NR if <15 yrs."""
    show_counts(hts, "UnknownStatusPartner")
    recode_yes_no(hts, "UnknownStatusPartner", nr_rules=[
        lambda d: (d["Age"] < 15) & d["UnknownStatusPartner"].notna(),
    ])
    show_counts(hts, "UnknownStatusPartner")
    positivity_table(hts, "UnknownStatusPartner")
    count_yes_without_sex(hts, "UnknownStatusPartner")
    return hts

In [ ]:
def clean_known_status_partner(hts):
    """KnownStatusPartner: unprotected sex with a partner of known status; NR if <15 yrs."""
    show_counts(hts, "KnownStatusPartner")
    recode_yes_no(hts, "KnownStatusPartner", nr_rules=[
        lambda d: (d["Age"] < 15) & d["KnownStatusPartner"].notna(),
    ])
    show_counts(hts, "KnownStatusPartner")
    positivity_table(hts, "KnownStatusPartner")
    count_yes_without_sex(hts, "KnownStatusPartner")
    return hts

### Key / priority population features

In [ ]:
def clean_population_type(hts):
    """PopulationType: NA fill, merge Vulnerable into Key Population, relabel GP / KP / PRIORITY."""
    hts["PopulationType"] = hts["PopulationType"].fillna("NA")
    show_counts(hts, "PopulationType")
    hts["PopulationType"] = hts["PopulationType"].replace("Vulnerable Population", "Key Population")
    hts["PopulationType"] = hts["PopulationType"].str.upper()
    mapping = {"GENERAL POPULATION": "GP", "KEY POPULATION": "KP",
               "PRIORITY POPULATION": "PRIORITY"}
    hts["PopulationType"] = hts["PopulationType"].replace(mapping)
    show_counts(hts, "PopulationType")
    return hts

In [ ]:
def clean_key_population(hts):
    """KeyPopulation: FSW / MSM / PRISONER / PWID (incl. OTHER); NR if not KP or <15 yrs."""
    show_counts(hts, "KeyPopulation")
    hts["KeyPopulation"] = hts["KeyPopulation"].str.upper()
    kp = hts["PopulationType"] == "KP"
    kpop = hts["KeyPopulation"]
    conditions = [
        hts["PopulationType"] != "KP",
        (hts["Age"] < 15) & kpop.notna(),
        kp & kpop.eq("FEMALE SEX WORKER"),
        kp & kpop.str.contains("MEN WHO HAVE SEX WITH MEN", na=False),
        kp & kpop.str.contains("PEOPLE IN PRISON AND OTHER CLOSED SETTINGS", na=False),
        kp & kpop.str.contains("PEOPLE WHO INJECT DRUGS|OTHER", na=False),
    ]
    choices = ["NR", "NR", "FSW", "MSM", "PRISONER", "PWID"]
    hts["KeyPopulation"] = select_labels(hts, conditions, choices)
    show_counts(hts, "KeyPopulation")
    positivity_table(hts, "KeyPopulation")             # higher in MSM
    return hts

In [ ]:
def clean_priority_population(hts):
    """PriorityPopulation: grouped into FISHERMEN / AGYW / PRISONERS / TRUCKDRIVERS /
    MILITARY / OTHER; NR if not PRIORITY or <15 yrs.
    (Uses the final, more granular version of the two passes in the original script.)"""
    show_counts(hts, "PriorityPopulation")
    hts["PriorityPopulation"] = hts["PriorityPopulation"].str.upper().str.strip()
    pp = hts["PriorityPopulation"]
    is_priority = hts["PopulationType"] == "PRIORITY"

    mapping = {
        "FISHER FOLK": "FISHERMEN",
        "ADOLESCENT AND YOUNG GIRLS": "AGYW",
        "YOUNG WOMEN AGED 15-24 YEARS": "AGYW",
        "PRISONER": "PRISONERS",
        "TRUCK DRIVER": "TRUCKDRIVERS",
        "MILITARY AND OTHER UNIFORMED SERVICES": "MILITARY",
        "FAMILIES AND CHILDREN LIVING ON THE STREETS": "OTHER",
        "PEOPLE WHO ABUSE ALCOHOL AND OTHER DRUGS": "OTHER",
        "SERVICE MEN AND WOMEN AND THEIR FAMILIES": "OTHER",
        "WIDOWS AND WIDOWERES": "OTHER",
        "OTHERS": "OTHER",
        "ORPHANS AND VULNERABLE CHILDREN": "OTHER",
    }
    conditions = [~is_priority, (hts["Age"] < 15) & pp.notna()]
    choices = ["NR", "NR"]
    for raw_label, new_label in mapping.items():
        conditions.append(is_priority & pp.eq(raw_label))
        choices.append(new_label)
    hts["PriorityPopulation"] = select_labels(hts, conditions, choices)
    show_counts(hts, "PriorityPopulation")
    crosstab_check(hts, "PriorityPopulation", TARGET)
    return hts

In [ ]:
def fix_population_type(hts):
    """Re-derive PopulationType: GP if both KP and Priority are NR, else KP, else PRIORITY."""
    conditions = [
        (hts["KeyPopulation"] == "NR") & (hts["PriorityPopulation"] == "NR"),
        hts["KeyPopulation"] != "NR",
        hts["PriorityPopulation"] != "NR",
    ]
    choices = ["GP", "KP", "PRIORITY"]
    hts["PopulationType"] = np.select(conditions, choices, default=hts["PopulationType"])
    show_counts(hts, "PopulationType")
    return hts

In [ ]:
def clean_is_health_worker(hts):
    """IsHealthWorker: NR if <17 yrs."""
    show_counts(hts, "IsHealthWorker")
    recode_yes_no(hts, "IsHealthWorker", allow_declined=False, nr_rules=[
        lambda d: (d["Age"] < 17) & d["IsHealthWorker"].notna(),
    ])
    show_counts(hts, "IsHealthWorker")
    positivity_table(hts, "IsHealthWorker")            # very low among health workers
    return hts

### TB risk features

In [ ]:
def clean_tb_status(hts):
    """TBStatus: NOTBSIGNS / PRESUMED / CONFIRMED / NOTDONE."""
    show_counts(hts, "TBStatus")
    hts["TBStatus"] = hts["TBStatus"].str.upper().str.strip()
    show_counts(hts, "TBStatus")
    conditions = [
        hts["TBStatus"].eq("NO TB SIGNS"),
        hts["TBStatus"].eq("PRESUMED TB"),
        hts["TBStatus"].eq("TB CONFIRMED"),
        hts["TBStatus"].eq("TB SCREENING NOT DONE"),
    ]
    choices = ["NOTBSIGNS", "PRESUMED", "CONFIRMED", "NOTDONE"]
    hts["TBStatus"] = select_labels(hts, conditions, choices)
    show_counts(hts, "TBStatus")
    positivity_table(hts, "TBStatus")                  # highest in confirmed TB
    return hts

In [ ]:
def clean_screened_tb(hts):
    """ScreenedTB: DECLINED / NO / YES."""
    show_counts(hts, "ScreenedTB")
    recode_yes_no(hts, "ScreenedTB")
    show_counts(hts, "ScreenedTB")
    positivity_table(hts, "ScreenedTB")
    return hts

### Violence and vulnerability features

In [ ]:
def clean_experienced_violence_screening(hts):
    """ExperiencedViolenceScreening: NR if <9 yrs."""
    show_counts(hts, "ExperiencedViolenceScreening")
    recode_yes_no(hts, "ExperiencedViolenceScreening", as_str=True, nr_rules=[
        lambda d: (d["Age"] < 9) & d["ExperiencedViolenceScreening"].notna(),
    ])
    positivity_table(hts, "ExperiencedViolenceScreening")
    return hts

In [ ]:
def gbv_proportion_tables(hts):
    """Row-normalised crosstabs of violence screening by sex, age and population type."""
    return {
        "by_sex": pd.crosstab(hts["Sex"], hts["ExperiencedViolenceScreening"], normalize="index"),
        "by_age": pd.crosstab(hts["Age"], hts["ExperiencedViolenceScreening"], normalize="index"),
        "by_population_type": pd.crosstab(hts["PopulationType"],
                                          hts["ExperiencedViolenceScreening"], normalize="index"),
    }

In [ ]:
def clean_violence_screening_type(hts):
    """Create GBVSexual / GBVPhysical / GBVEmotional flags from ViolenceScreeningType."""
    hts["ViolenceScreeningType"] = hts["ViolenceScreeningType"].str.upper()
    vst = hts["ViolenceScreeningType"]
    for new_col, keyword in [("GBVSexual", "SEXUAL"),
                             ("GBVPhysical", "PHYSICAL"),
                             ("GBVEmotional", "EMOTIONAL")]:
        conditions = [
            vst.str.contains(keyword, na=False),
            hts["ExperiencedViolenceScreening"] == "NO",
            ~vst.isin(["NULL", "", "NAN"]),
        ]
        hts[new_col] = np.select(conditions, ["YES", "NO", "NAN"], default="NR")
    return hts

In [ ]:
def check_gbv_consistency(hts):
    """Consistency checks between violence screening and the GBV type flags."""
    for flag in ["GBVEmotional", "GBVPhysical", "GBVSexual"]:
        crosstab_check(hts, "ExperiencedViolenceScreening", flag)

    n_no_gbv_but_flags = hts[
        (hts["ExperiencedViolenceScreening"].isna() | (hts["ExperiencedViolenceScreening"] == "NO")) &
        (hts["GBVEmotional"] == "YES") & (hts["GBVPhysical"] == "YES") & (hts["GBVSexual"] == "YES")
    ].shape[0]
    print("Number of individuals who did not experience GBV (or status is NA) "
          f"and GBV types are 'YES': {n_no_gbv_but_flags}")

    n_yes_no_type = hts[
        (hts["ExperiencedViolenceScreening"] == "YES") &
        (hts["GBVEmotional"] != "YES") & (hts["GBVPhysical"] != "YES") & (hts["GBVSexual"] != "YES")
    ].shape[0]
    print("Number of individuals who experienced GBV but did not report any "
          f"specific GBV type: {n_yes_no_type}")
    return n_no_gbv_but_flags, n_yes_no_type

In [ ]:
def report_gbv_positivity(hts):
    """Positivity tables for violence screening and each GBV type."""
    return {col: positivity_table(hts, col)
            for col in ["ExperiencedViolenceScreening", "GBVEmotional", "GBVSexual", "GBVPhysical"]}

In [ ]:
def clean_patient_disabled(hts):
    """PatientDisabled: standardise to upper case and review positivity."""
    show_counts(hts, "PatientDisabled")
    hts["PatientDisabled"] = hts["PatientDisabled"].str.upper()
    crosstab_check(hts, "PatientDisabled", "PopulationType")
    positivity_table(hts, "PatientDisabled")           # lower in disabled patients
    return hts

### PrEP features

In [ ]:
def clean_currently_on_prep(hts):
    """CurrentlyOnPrep: NR if <15 yrs."""
    show_counts(hts, "CurrentlyOnPrep")
    recode_yes_no(hts, "CurrentlyOnPrep", nr_rules=[
        lambda d: (d["Age"] < 15) & d["CurrentlyOnPrep"].notna(),
    ])
    show_counts(hts, "CurrentlyOnPrep")
    positivity_table(hts, "CurrentlyOnPrep")
    return hts

In [ ]:
def prep_by_sex(hts):
    """Proportion currently on PrEP within each sex."""
    return hts.groupby("Sex")["CurrentlyOnPrep"].value_counts(normalize=True).unstack(fill_value=0)

In [ ]:
def service_flag(df, keyword):
    """YES if `keyword` appears in ReceivedServices, NO if services recorded but not keyword, else None."""
    return np.select(
        [df["ReceivedServices"].str.contains(rf"(?:^|,){keyword}(?:,|$)", na=False),
         df["ReceivedServices"].notna()],
        ["YES", "NO"],
        default=None,
    )

In [ ]:
def clean_received_services(hts, services=("PREP", "PEP", "TB", "STI")):
    """Create ReceivedPREP / ReceivedPEP / ReceivedTB / ReceivedSTI binaries."""
    show_counts(hts, "ReceivedServices")
    hts["ReceivedServices"] = hts["ReceivedServices"].str.upper()
    for service in services:
        hts[f"Received{service}"] = service_flag(hts, service)
    for service in services:
        print(f"\n{service}")
        show_counts(hts, f"Received{service}")
    for service in services:
        positivity_table(hts, f"Received{service}")
    return hts

In [ ]:
def check_services_consistency(hts):
    """Quick checks: PrEP services vs currently on PrEP; TB services vs TB status."""
    return (crosstab_check(hts, "ReceivedPREP", "CurrentlyOnPrep"),
            crosstab_check(hts, "ReceivedTB", "TBStatus"))

### Sexual exposure features

In [ ]:
def clean_ever_had_sex(hts):
    """EverHadSex: NR if <15 yrs, otherwise YES / NO."""
    show_counts(hts, "EverHadSex")
    hts["EverHadSex"] = hts["EverHadSex"].str.upper()
    conditions = [
        hts["Age"] < 15,
        hts["EverHadSex"].str.contains("YES", na=False),
        hts["EverHadSex"].str.contains("NO", na=False),
    ]
    hts["EverHadSex"] = select_labels(hts, conditions, ["NR", "YES", "NO"])
    show_counts(hts, "EverHadSex")
    positivity_table(hts, "EverHadSex")
    return hts

In [ ]:
def clean_sexually_active(hts):
    """SexuallyActive: NR if <15 yrs."""
    show_counts(hts, "SexuallyActive")
    recode_yes_no(hts, "SexuallyActive", as_str=True, nr_rules=[
        lambda d: (d["Age"] < 15) & d["SexuallyActive"].notna(),
    ])
    show_counts(hts, "SexuallyActive")
    # missing values become the string 'NAN' here, matching the original behaviour
    hts["SexuallyActive"] = hts["SexuallyActive"].astype(str).str.upper()
    positivity_table(hts, "SexuallyActive")
    return hts

### Traditional exposure / family risk / time features

In [ ]:
def clean_traditional_procedures(hts):
    """TraditionalProcedures: standardise to upper case."""
    show_counts(hts, "TraditionalProcedures")
    hts["TraditionalProcedures"] = hts["TraditionalProcedures"].str.upper()
    positivity_table(hts, "TraditionalProcedures")
    return hts

In [ ]:
def clean_mothers_status(hts):
    """MothersStatus (asked of children only): NR for adults with no value."""
    show_counts(hts, "MothersStatus")
    hts["MothersStatus"] = hts["MothersStatus"].str.upper()
    ms = hts["MothersStatus"]
    conditions = [
        (hts["Age"] >= 15) & ms.isna(),
        ms.str.contains("POSITIVE", na=False),
        ms.str.contains("NEGATIVE", na=False),
        ms.str.contains("UNKNOWN", na=False),
    ]
    hts["MothersStatus"] = select_labels(hts, conditions, ["NR", "POSITIVE", "NEGATIVE", "UNKNOWN"])
    show_counts(hts, "MothersStatus")
    positivity_table(hts, "MothersStatus")
    return hts

In [ ]:
def add_day_of_week(hts):
    """Day of week of the visit (month skipped: data does not cover a full year)."""
    hts["dayofweek"] = hts["VDate"].dt.day_name().str.upper()
    show_counts(hts, "dayofweek")
    positivity_table(hts, "dayofweek")                 # higher on Monday
    return hts

### Finalise Stage 2

In [ ]:
def drop_unneeded_columns(hts, cols_to_drop=FINAL_DROP_COLS):
    """Drop helper / redundant columns before saving the cleaned dataset."""
    hts = hts.drop(columns=cols_to_drop, errors="ignore")
    print_shape(hts, "hts (after dropping columns)")
    return hts

In [ ]:
def report_missingness(hts):
    """Missing count and % for every column."""
    missing = hts.isnull().sum()
    df = pd.DataFrame({"Missing Values": missing,
                       "Percentage": missing / len(hts) * 100})
    if VERBOSE:
        display(df)
    return df

In [ ]:
def report_variance(hts, cols=VARIANCE_CHECK_COLS):
    """Print the % distribution of each variable to spot no/low-variance columns."""
    for col in [c for c in cols if c in hts.columns]:
        print(f"\n{'=' * 40}")
        print(f"Column: {col}")
        print(hts[col].value_counts(dropna=False, normalize=True) * 100)

In [ ]:
def drop_mfl_code(hts):
    """Drop MFL_Code (redundant with SiteCode)."""
    return hts.drop(columns=["MFL_Code"], errors="ignore")

In [ ]:
def run_stage2_clean_and_engineer(hts=None, paths=PATHS, download=False):
    """Run all of Stage 2 in the original order and save hts_cleaned.csv."""
    if hts is None:
        hts = load_hts_raw(paths["hts_raw"])

    # Demographics
    hts = add_age_features(hts)
    hts = clean_sex(hts)
    hts = drop_missing_test_result(hts)
    hts = clean_marital_status(hts)
    # Clinical risk
    hts = clean_pregnant(hts)
    hts = clean_breastfeeding_mother(hts)
    # HIV testing history
    hts = clean_tested_hiv_before(hts)
    hts = clean_months_since_last_test(hts)
    # Behaviour risk
    hts = clean_new_partner(hts)
    check_sexually_active_vs_new_partner(hts)
    hts = clean_multiple_partners(hts)
    check_multiple_partners_vs_sexually_active(hts)
    hts = clean_money_sex(hts)
    hts = clean_alcohol_sex(hts)
    hts = clean_condom_burst(hts)
    # Partner risk
    hts = clean_partner_hiv_status(hts)
    hts = clean_unknown_status_partner(hts)
    hts = clean_known_status_partner(hts)
    # Key / priority population
    hts = clean_population_type(hts)
    hts = clean_key_population(hts)
    hts = clean_priority_population(hts)
    hts = fix_population_type(hts)
    hts = clean_is_health_worker(hts)
    # TB
    hts = clean_tb_status(hts)
    hts = clean_screened_tb(hts)
    # Violence / vulnerability
    hts = clean_experienced_violence_screening(hts)
    hts = clean_violence_screening_type(hts)
    check_gbv_consistency(hts)
    report_gbv_positivity(hts)
    hts = clean_patient_disabled(hts)
    # PrEP and services
    hts = clean_currently_on_prep(hts)
    hts = clean_received_services(hts)
    check_services_consistency(hts)
    # Sexual exposure
    hts = clean_ever_had_sex(hts)
    hts = clean_sexually_active(hts)
    # Traditional exposure, family, time
    hts = clean_traditional_procedures(hts)
    hts = clean_mothers_status(hts)
    hts = add_day_of_week(hts)

    # Finalise
    hts = drop_unneeded_columns(hts)
    save_csv(hts, paths["hts_cleaned"], download=download)
    report_missingness(hts)
    report_variance(hts)
    hts = drop_mfl_code(hts)
    return hts

## STAGE 3 - GIS JOIN, TRAIN/VAL/TEST SPLIT, IMPUTATION

In [ ]:
def load_cleaned_data(path=PATHS["hts_cleaned"]):
    """Load the cleaned HTS dataset produced by Stage 2."""
    return pd.read_csv(path, low_memory=False)

In [ ]:
def load_gis_features(path=PATHS["gis"]):
    """Read the GIS .rds file (needs: pip install pyreadr), drop lat/long and duplicates,
    and mean-impute numeric GIS columns."""
    import pyreadr
    gis = pyreadr.read_r(path)[None]
    if VERBOSE:
        display(gis.head())
    gis = gis.drop(columns=[c for c in ["Latitude", "Longitude"] if c in gis.columns])
    gis = gis.drop_duplicates(subset=["FacilityCode"], keep="first")
    numeric_cols = gis.select_dtypes(include=np.number).columns
    gis[numeric_cols] = gis[numeric_cols].fillna(gis[numeric_cols].mean())
    return gis

In [ ]:
def join_gis_features(hts, gis):
    """Inner-join GIS variables on SiteCode = FacilityCode."""
    hts = hts.copy()
    hts["SiteCode"] = hts["SiteCode"].astype(str)
    hts = pd.merge(hts, gis, left_on="SiteCode", right_on="FacilityCode", how="inner")
    hts = hts.replace("", np.nan)
    print_shape(hts, "hts (after joining GIS variables)")
    return hts

In [ ]:
def set_random_seeds(seed=SEED):
    """Set Python and NumPy seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)

In [ ]:
def train_test_split_stratified(df, test_size, seed=SEED):
    """Thin wrapper around sklearn's train_test_split, stratified on the target."""
    from sklearn.model_selection import train_test_split
    return train_test_split(df, test_size=test_size, stratify=df[TARGET], random_state=seed)

In [ ]:
def split_train_val_test(hts, seed=SEED):
    """Stratified 60/20/20 train/validation/test split on the target."""
    train, temp = train_test_split_stratified(hts, test_size=0.4, seed=seed)
    val, test = train_test_split_stratified(temp, test_size=0.5, seed=seed)
    return train, val, test

In [ ]:
def build_sparse_datasets(train, val, test):
    """Sparse versions keep missing values missing (e.g. for XGBoost)."""
    sparse = {"sparse_train": train.copy(), "sparse_val": val.copy(), "sparse_test": test.copy()}
    print("Sparse:")
    for name, df in sparse.items():
        print(f"  {name} shape: {df.shape}")
    return sparse

In [ ]:
def save_sparse_datasets(sparse, download=False):
    """Save each sparse dataset to CSV and the dictionary to sparse_dict.pkl."""
    for name, df in sparse.items():
        save_csv(df, f"{name}.csv", download=download)
    with open("sparse_dict.pkl", "wb") as f:
        pickle.dump(sparse, f)
    if download:
        download_file("sparse_dict.pkl")

In [ ]:
def identify_imputation_columns(hts, first_block=slice(0, 35), second_block=slice(36, 48),
                                present_threshold=50):
    """Split columns into those to impute (>50% present, some missing) and those to label
    MISSING (>50% missing). The default column slices follow the original script;
    adjust them if your column layout changes."""
    cols_to_impute, cols_to_unknown = [], []
    columns = list(hts.columns[first_block]) + list(hts.columns[second_block])
    for col in columns:
        percent_present = hts[col].notna().mean() * 100
        percent_missing = hts[col].isna().mean() * 100
        if percent_present > present_threshold and hts[col].isna().any():
            cols_to_impute.append(col)
        if percent_missing > present_threshold:
            cols_to_unknown.append(col)
    print(f"\nColumns to impute: {cols_to_impute}")
    print(f"Columns to label as unknown: {cols_to_unknown}")
    return cols_to_impute, cols_to_unknown

In [ ]:
def mode_excluding_nr(series):
    """Most frequent value ignoring 'NR' and NaN (returns None if nothing valid)."""
    valid_values = series[(series != "NR") & (series.notna())]
    if valid_values.empty:
        return None
    return valid_values.mode().iloc[0]

In [ ]:
def impute_with_train_mode(train_df, val_df, test_df, cols_to_impute):
    """Fill missing values with the TRAIN mode (no leakage) in all three sets."""
    for col in cols_to_impute:
        mode_value = mode_excluding_nr(train_df[col])
        if mode_value is None:
            print(f"Skipping {col}: no valid mode found")
            continue
        for df in (train_df, val_df, test_df):
            df[col] = df[col].fillna(mode_value)
    return train_df, val_df, test_df

In [ ]:
def label_sparse_columns_missing(dfs, cols_to_unknown):
    """For very sparse columns, label missing values as 'MISSING'."""
    for col in cols_to_unknown:
        for df in dfs:
            df[col] = df[col].fillna("MISSING")
    return dfs

In [ ]:
def add_missing_indicators(simple_df, sparse_df, sparse_cols):
    """Append *_bin flags (1 = value present, 0 = missing) for columns that had missingness
    in the sparse TRAIN set."""
    flags = sparse_df[sparse_cols].notna().astype(int)
    flags.columns = [f"{col}_bin" for col in sparse_cols]
    return pd.concat([simple_df, flags], axis=1)

In [ ]:
def build_simple_datasets(hts, train, val, test, sparse):
    """Create the simple-imputed train/val/test datasets with missingness indicators."""
    cols_to_impute, cols_to_unknown = identify_imputation_columns(hts)

    train_simple, val_simple, test_simple = train.copy(), val.copy(), test.copy()
    train_simple, val_simple, test_simple = impute_with_train_mode(
        train_simple, val_simple, test_simple, cols_to_impute)
    label_sparse_columns_missing([train_simple, val_simple, test_simple], cols_to_unknown)

    sparse_cols = sparse["sparse_train"].columns[sparse["sparse_train"].isna().any()].tolist()
    train_simple = add_missing_indicators(train_simple, sparse["sparse_train"], sparse_cols)
    val_simple = add_missing_indicators(val_simple, sparse["sparse_val"], sparse_cols)
    test_simple = add_missing_indicators(test_simple, sparse["sparse_test"], sparse_cols)
    return train_simple, val_simple, test_simple

In [ ]:
def save_simple_datasets(train_simple, val_simple, test_simple, download=False):
    """Save the simple-imputed datasets to CSV."""
    save_csv(train_simple, "train_simple.csv", download=download)
    save_csv(val_simple, "val_simple.csv", download=download)
    save_csv(test_simple, "test_simple.csv", download=download)

In [ ]:
def run_stage3_impute_and_split(hts=None, paths=PATHS, download=False):
    """Run all of Stage 3: load cleaned data, join GIS, split, build sparse and simple datasets."""
    if hts is None:
        hts = load_cleaned_data(paths["hts_cleaned"])
    gis = load_gis_features(paths["gis"])
    hts = join_gis_features(hts, gis)

    set_random_seeds()
    train, val, test = split_train_val_test(hts)

    sparse = build_sparse_datasets(train, val, test)
    save_sparse_datasets(sparse, download=download)

    train_simple, val_simple, test_simple = build_simple_datasets(hts, train, val, test, sparse)
    save_simple_datasets(train_simple, val_simple, test_simple, download=download)
    return sparse, (train_simple, val_simple, test_simple)

## STAGE 4 - POSITIVITY OVER TIME

In [ ]:
def compute_monthly_positivity(hts):
    """Monthly tested, positives and positivity rate (%)."""
    df = hts.dropna(subset=[TARGET]).copy()
    df["VisitDate"] = pd.to_datetime(df["VisitDate"])
    df["YearMonth"] = df["VisitDate"].dt.to_period("M")
    monthly = df.groupby("YearMonth").agg(
        Total_Tested=(TARGET, "count"),
        Positive=(TARGET, lambda x: (x.astype(str).str.upper() == "POSITIVE").sum()),
    )
    monthly["Positivity_Rate"] = monthly["Positive"] / monthly["Total_Tested"] * 100
    return monthly

In [ ]:
def plot_monthly_positivity(monthly, title="HIV Positivity Rate (April 2025 - June 2026)"):
    """Line plot of monthly positivity rate."""
    import matplotlib.pyplot as plt
    plt.figure(figsize=(12, 5))
    monthly["Positivity_Rate"].plot(marker="o")
    plt.ylabel("Positivity Rate (%)")
    plt.xlabel("Month")
    plt.title(title)
    plt.grid(True)
    plt.show()

In [ ]:
def monthly_unique_counts(hts, column):
    """Number of unique values of `column` (e.g. County, SiteCode) per visit month."""
    months = pd.to_datetime(hts["VisitDate"]).dt.to_period("M")
    return hts.groupby(months)[column].nunique()

In [ ]:
def run_stage4_positivity_over_time(hts=None, paths=PATHS):
    """Run all of Stage 4 on the cleaned dataset."""
    if hts is None:
        hts = load_cleaned_data(paths["hts_cleaned"])
    show_counts(hts, TARGET)
    monthly = compute_monthly_positivity(hts)
    display(monthly)
    plot_monthly_positivity(monthly)
    display(monthly_unique_counts(hts, "County"))
    display(monthly_unique_counts(hts, "SiteCode"))
    return monthly

## MAIN

In [ ]:
def main(download=False):
    """End-to-end run of all four stages."""
    run_stage1_combine_and_filter(download=download)
    run_stage2_clean_and_engineer(download=download)
    run_stage3_impute_and_split(download=download)
    run_stage4_positivity_over_time()

## Run the pipeline

Set `download=True` to trigger Colab downloads of the saved CSVs. Each stage can be run independently because Stage 1 → 2 → 3 pass data through files (`hts.csv` → `hts_cleaned.csv`).

### Stage 1: combine and filter

In [ ]:
hts = run_stage1_combine_and_filter(download=False)

### Stage 2: clean and engineer features

In [ ]:
hts_clean = run_stage2_clean_and_engineer(download=False)

### Stage 3: GIS join, split and imputation

In [ ]:
sparse, (train_simple, val_simple, test_simple) = run_stage3_impute_and_split(download=False)

### Stage 4: positivity over time

In [ ]:
monthly = run_stage4_positivity_over_time()